In [ ]:
!pip install deep-translator tqdm
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.4 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import re

from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

import torch
from transformers import MarianMTModel, MarianTokenizer

from tqdm import tqdm

from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
)

In [ ]:
fakes = pd.read_csv("Fake.csv")
trues = pd.read_csv("True.csv")

trues["label"] = 1
fakes["label"] = 0

data = pd.concat([trues, fakes])

In [ ]:
model_name = "Helsinki-NLP/opus-mt-en-ru"
tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

def translate_batch(texts_list):
    inputs = tokenizer(texts_list, return_tensors="pt", padding=True, truncation=True, max_length=128).to(device)
    translated = model.generate(**inputs)
    return [tokenizer.decode(t, skip_special_tokens=True) for t in translated]

batch_size = 32
translated_titles = []

titles = data["title"].tolist()

for i in tqdm(range(0, len(titles), batch_size)):
    batch = titles[i:i + batch_size]
    batch = [str(text) if pd.notna(text) else " " for text in batch]

    translated_batch = translate_batch(batch)
    translated_titles.extend(translated_batch)

data["title_ru"] = translated_titles

data.to_csv("russian_news_dataset.csv", index=False)

/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

100%|██████████| 1404/1404 [31:51<00:00,  1.36s/it]


In [ ]:
df = pd.read_csv("russian_news_dataset.csv")

df = df.dropna(subset=["title_ru", "label"]).reset_index(drop=True)

In [ ]:
df["title_len"] = df["title_ru"].astype(str).str.len()
df["exclamation_marks"] = df["title_ru"].astype(str).str.count("!")
df["question_marks"] = df["title_ru"].astype(str).str.count(r"\?")

def get_caps_ratio(text):
  text = str(text)
  if len(text) == 0:
    return 0.0

  letters = [c for c in text if c.isalpha()]

  if len(letters) == 0:
    return 0.0

  caps = [c for c in text if c.isupper()]

  return len(caps) / len(letters)

def get_caps_words_count(text):
  text = str(text)
  if len(text) == 0:
    return 0.0

  words = re.findall(r"\b\w+\b", text)

  caps = [c for c in words if c.isupper() and len(c) > 1]

  return len(caps)


df["caps_ratio"] = df["title_ru"].apply(get_caps_ratio)
df["caps_count"] = df["title_ru"].apply(get_caps_words_count)

In [ ]:
X = df[["title_ru", "exclamation_marks", "question_marks", "caps_ratio", "caps_count", "title_len"]]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
model = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.03,
    depth=6,
    eval_metric="F1",
    random_seed=42,
    verbose=100,
)

train_pool = Pool(X_train, y_train, text_features=["title_ru"])
test_pool = Pool(X_test, y_test, text_features=["title_ru"])

model.fit(train_pool, eval_set=test_pool)

cb_preds_prob = model.predict_proba(test_pool)[:, 1]
print("\nROC-AUC:", roc_auc_score(y_test, cb_preds_prob))

In [ ]:
model.save_model("catboost_model.cbm")

In [ ]:
MODEL_NAME = "cointegrated/rubert-tiny2"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def tokenize_data(data):
    return tokenizer(
        data["title_ru"].tolist(),
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors="pt",
    )


class Dataset(torch.utils.data.Dataset):
    def __init__(self, data, labels):
        self.data = data
        self.labels = labels.values

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.data.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.int16)
        return item

    def __len__(self):
        return len(self.labels)


train_dataset = Dataset(tokenize_data(X_train), y_train)
test_dataset = Dataset(tokenize_data(X_test), y_test)

bert_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2
)

training_args = TrainingArguments(
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    logging_steps=100,
    learning_rate=3e-5,
)

trainer = Trainer(
    model=bert_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
)

trainer.train()

config.json:   0%|          | 0.00/693 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/401 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/1.08M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.74M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/55 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss
100,0.449552
200,0.231062
300,0.202494
400,0.165702
500,0.142398
600,0.163887
700,0.144694
800,0.133291
900,0.144958
1000,0.135674


TrainOutput(global_step=3369, training_loss=0.1149490634706474, metrics={'train_runtime': 138.8411, 'train_samples_per_second': 776.096, 'train_steps_per_second': 24.265, 'total_flos': 198650074622976.0, 'train_loss': 0.1149490634706474, 'epoch': 3.0})

In [ ]:
bert_model.save_pretrained("rubert_model")
tokenizer.save_pretrained("rubert_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('rubert_model/tokenizer_config.json', 'rubert_model/tokenizer.json')